In [ ]:
# analysis/plot_results.py

from pathlib import Path
import ast

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    accuracy_score,
)


# ---------------------------------------------------------
# Utility
# ---------------------------------------------------------

def parse_path_column(series):
    return series.apply(
        lambda x: ast.literal_eval(x)
        if isinstance(x, str)
        else x
    )


# ---------------------------------------------------------
# Training curves
# ---------------------------------------------------------

def plot_training_curves(metrics_csv_path, output_dir):

    metrics_csv_path = Path(metrics_csv_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(metrics_csv_path)

    # ---------------- LOSS ----------------
    plt.figure()

    plt.plot(df["epoch"], df["train_loss"], label="train")
    plt.plot(df["epoch"], df["val_loss"], label="val")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss")

    plt.legend()
    plt.tight_layout()

    plt.savefig(output_dir / "loss_curve.png", dpi=200)
    plt.close()

    # ---------------- ACCURACY ----------------
    plt.figure()

    if "val_path_acc" in df:
        plt.plot(df["epoch"], df["val_path_acc"], label="path_acc")

    if "val_leaf_acc" in df:
        plt.plot(df["epoch"], df["val_leaf_acc"], label="leaf_acc")

    level_cols = [
        c for c in df.columns
        if c.startswith("val_level_")
        and c.endswith("_acc")
    ]

    for col in level_cols:
        plt.plot(df["epoch"], df[col], label=col)

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Validation Accuracy")

    plt.legend()
    plt.tight_layout()

    plt.savefig(output_dir / "accuracy_curve.png", dpi=200)
    plt.close()


# ---------------------------------------------------------
# Hierarchical metrics
# ---------------------------------------------------------

def compute_level_metrics(
    predictions_df,
    hierarchy_tree,
):

    metrics = {}

    depth = hierarchy_tree.total_depth

    for level in range(depth):

        y_true = predictions_df["target_path"].apply(
            lambda x: x[level]
        )

        y_pred = predictions_df["pred_path"].apply(
            lambda x: x[level]
        )

        metrics[level] = {
            "accuracy": accuracy_score(y_true, y_pred),
            "f1_macro": f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),
            "f1_weighted": f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            )
        }

    return metrics


# ---------------------------------------------------------
# Confusion matrix for arbitrary hierarchy level
# ---------------------------------------------------------

def plot_level_confusion_matrix(
    predictions_csv_path,
    hierarchy_tree,
    level,
    output_dir,
    epoch=None,
    normalize="true",
):

    predictions_csv_path = Path(predictions_csv_path)
    output_dir = Path(output_dir)

    df = pd.read_csv(predictions_csv_path)

    if epoch is None:
        epoch = df["epoch"].max()

    df = df[df["epoch"] == epoch].copy()

    df["pred_path"] = parse_path_column(df["pred_path"])
    df["target_path"] = parse_path_column(df["target_path"])

    y_true = df["target_path"].apply(lambda x: x[level]).to_numpy()
    y_pred = df["pred_path"].apply(lambda x: x[level]).to_numpy()

    labels = sorted(set(y_true) | set(y_pred))

    class_names = [
        hierarchy_tree.get_name_from_level_idx(level, idx)
        for idx in labels
    ]

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels,
        normalize=normalize,
    )

    fig, ax = plt.subplots(figsize=(10, 10))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names,
    )

    disp.plot(
        ax=ax,
        xticks_rotation=90,
        values_format=".2f",
        colorbar=True,
    )

    ax.set_title(
        f"Hierarchy Level {level} Confusion Matrix"
    )

    plt.tight_layout()

    plt.savefig(
        output_dir / f"confusion_matrix_level_{level}.png",
        dpi=200
    )

    plt.close()


# ---------------------------------------------------------
# Classification report
# ---------------------------------------------------------

def save_leaf_classification_report(
    predictions_df,
    hierarchy_tree,
    output_dir,
):

    leaf_level = hierarchy_tree.total_depth - 1

    y_true = predictions_df["target_path"].apply(
        lambda x: x[leaf_level]
    )

    y_pred = predictions_df["pred_path"].apply(
        lambda x: x[leaf_level]
    )

    target_names = [
        hierarchy_tree.get_name_from_level_idx(
            leaf_level,
            idx
        )
        for idx in sorted(set(y_true))
    ]

    report = classification_report(
        y_true,
        y_pred,
        target_names=target_names,
        output_dict=True,
        zero_division=0,
    )

    pd.DataFrame(report).transpose().to_csv(
        output_dir / "leaf_classification_report.csv"
    )


# ---------------------------------------------------------
# Main pipeline
# ---------------------------------------------------------

def make_all_plots(
    experiment_dir,
    hierarchy_tree,
):

    experiment_dir = Path(experiment_dir)

    metrics_csv = experiment_dir / "metrics.csv"
    predictions_csv = experiment_dir / "predictions.csv"

    output_dir = experiment_dir / "plots"
    output_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------------------------------
    # Load predictions
    # -------------------------------------------------

    pred_df = pd.read_csv(predictions_csv)

    pred_df["pred_path"] = parse_path_column(
        pred_df["pred_path"]
    )

    pred_df["target_path"] = parse_path_column(
        pred_df["target_path"]
    )

    # -------------------------------------------------
    # Curves
    # -------------------------------------------------

    plot_training_curves(
        metrics_csv,
        output_dir,
    )

    # -------------------------------------------------
    # Per-level confusion matrices
    # -------------------------------------------------

    for level in range(hierarchy_tree.total_depth):

        plot_level_confusion_matrix(
            predictions_csv_path=predictions_csv,
            hierarchy_tree=hierarchy_tree,
            level=level,
            output_dir=output_dir,
        )

    # -------------------------------------------------
    # Per-level metrics
    # -------------------------------------------------

    level_metrics = compute_level_metrics(
        pred_df,
        hierarchy_tree,
    )

    rows = []

    for level, stats in level_metrics.items():

        row = {
            "level": level,
            "accuracy": stats["accuracy"],
            "f1_macro": stats["f1_macro"],
            "f1_weighted": stats["f1_weighted"],
        }

        rows.append(row)

    pd.DataFrame(rows).to_csv(
        output_dir / "level_metrics.csv",
        index=False,
    )

    # -------------------------------------------------
    # Leaf report
    # -------------------------------------------------

    save_leaf_classification_report(
        pred_df,
        hierarchy_tree,
        output_dir,
    )

    print("Saved analysis to:", output_dir)

In [ ]:
from hierarchies.hierarchies import get_hierarchy_tree


tree = get_hierarchy_tree("urbansound8k")

make_all_plots(
    experiment_dir="outputs/experiment_1",
    hierarchy_tree=tree,
)

# Optional:
plot_level_confusion_matrix(
    predictions_csv_path="outputs/experiment_1/predictions.csv",
    hierarchy_tree=tree,
    level=1,
    output_dir="outputs/experiment_1/plots",
)